# Pandas — Datetime Operations

In [1]:
# Initialize

#Imports
import pandas as pd
pd.set_option("display.max_columns", None)
# Load Data
employees = pd.read_csv("../data/employees.csv")
departments = pd.read_csv("../data/departments.csv")
sales = pd.read_csv("../data/sales.csv")

In [2]:
# Convert string columns to datetime with error handling

employees["joining_date"] = pd.to_datetime(
    employees["joining_date"],
    errors="coerce"
)

sales["sale_date"] = pd.to_datetime(
    sales["sale_date"],
    errors="coerce"
)

employees.dtypes

emp_id                   int64
name                    object
department              object
salary                   int64
joining_date    datetime64[ns]
dtype: object

In [3]:
# Extract useful components

sales["year"] = sales["sale_date"].dt.year
sales["month"] = sales["sale_date"].dt.month
sales["day"] = sales["sale_date"].dt.day
sales["day_of_week"] = sales["sale_date"].dt.day_name()

sales.head()

,sale_id,emp_id,amount,sale_date,region,year,month,day,day_of_week
0,1,1,12000,2023-01-10,North,2023,1,10,Tuesday
1,2,2,15000,2023-01-12,East,2023,1,12,Thursday
2,3,5,8000,2023-02-01,West,2023,2,1,Wednesday
3,4,1,7000,2023-02-05,North,2023,2,5,Sunday
4,5,3,4000,2023-02-10,South,2023,2,10,Friday


In [4]:
# Create month bucket properly

sales["sale_month"] = sales["sale_date"].dt.to_period("M")

monthly_revenue = (
    sales.groupby("sale_month")
         .agg(total_revenue=("amount", "sum"))
         .reset_index()
)

monthly_revenue

,sale_month,total_revenue
0,2023-01,27000
1,2023-02,19000
2,2023-03,58000
3,2023-04,50500
4,2023-05,43500
5,2023-06,58500


In [5]:
# Calculate employee tenure in days

current_date = pd.Timestamp("2026-02-22")

employees["tenure_days"] = (
    current_date - employees["joining_date"]
).dt.days

employees.head()

,emp_id,name,department,salary,joining_date,tenure_days
0,1,Amit,Engineering,80000,2021-06-15,1713
1,2,Neha,Engineering,75000,2022-03-10,1445
2,3,Ravi,HR,50000,2020-01-20,2225
3,4,Pooja,HR,55000,2021-11-05,1570
4,5,Karan,Sales,60000,2023-02-01,1117


In [13]:
# Sort before calculating time differences
sales = sales.sort_values(["emp_id", "sale_date"])

sales["previous_sale_date"] = (
    sales.groupby("emp_id")["sale_date"]
         .shift(1)
)

sales["days_since_last_sale"] = (
    sales["sale_date"] - sales["previous_sale_date"]
).dt.days

sales.head()

,sale_id,emp_id,amount,sale_date,region,year,month,day,day_of_week,sale_month,previous_sale_date,days_since_last_sale
0,1,1,12000,2023-01-10,North,2023,1,10,Tuesday,2023-01,NaT,NaN
3,4,1,7000,2023-02-05,North,2023,2,5,Sunday,2023-02,2023-01-10,26.0
14,15,1,10000,2023-05-05,North,2023,5,5,Friday,2023-05,2023-02-05,89.0
1,2,2,15000,2023-01-12,East,2023,1,12,Thursday,2023-01,NaT,NaN
13,14,2,15500,2023-05-01,East,2023,5,1,Monday,2023-05,2023-01-12,109.0


In [16]:
# Simulating incremental filter

last_processed_date = pd.Timestamp("2023-02-22")

incremental_sales = sales[
    sales["sale_date"] > last_processed_date
]

incremental_sales.head()

,sale_id,emp_id,amount,sale_date,region,year,month,day,day_of_week,sale_month,previous_sale_date,days_since_last_sale
14,15,1,10000,2023-05-05,North,2023,5,5,Friday,2023-05,2023-02-05,89.0
13,14,2,15500,2023-05-01,East,2023,5,1,Monday,2023-05,2023-01-12,109.0
16,17,5,8500,2023-05-15,South,2023,5,15,Monday,2023-05,2023-02-01,103.0
9,10,6,7500,2023-04-01,East,2023,4,1,Saturday,2023-04,NaT,NaN
5,6,7,20000,2023-03-01,East,2023,3,1,Wednesday,2023-03,NaT,NaN
